# M5A1 - Inspeção Visual de Itens em Esteira de Manufatura

Na prática de hoje vamos refinar um modelo para a tarefa de inspeção visual.

Esse notebook está estruturado da seguinte forma.

- Introdução
- Carregar Base de Dados
- Refinar Modelo
- Próximos passos
- Atividade Complementares

## Introdução

Instalação para os que ainda não possuem a biblioteca instalada.

In [3]:
!pip install torch torchvision huggingface_hub ultralytics

Importar as bibliotecas

In [4]:
import yaml
import os

from ultralytics import YOLO
from huggingface_hub import snapshot_download
from pathlib import Path
import shutil
from IPython.display import Video

/home/joaoferreira/trilha_visao_computacional/.env-trilha-vc/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Carregar Base de Dados

A primeira tarefa para refinar um modelo é criar a base de dados.

Referência: https://huggingface.co/datasets/johnatanvq/fruits-dataset

In [5]:
# 1) Baixar somente o subdiretório fruitsData/
local_repo_dir = snapshot_download(
    repo_id="johnatanvq/fruits-dataset",
    repo_type="dataset",
    allow_patterns=["fruitsData/**"],  # baixa só essa pasta
)

print("Arquivos baixados em:", local_repo_dir)

# 2) Mover/copiar para uma pasta final estilo ImageFolder (se quiser customizar o caminho)
src = Path(local_repo_dir) / "fruitsData"
dst = Path("data/fruits")  # pasta final onde você quer o ImageFolder

# Copiando arquivos para dst.
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(src, dst)

print("ImageFolder pronto em:", dst)


Fetching 322 files: 100%|██████████| 322/322 [00:00<00:00, 13243.96it/s]


Arquivos baixados em: /home/joaoferreira/.cache/huggingface/hub/datasets--johnatanvq--fruits-dataset/snapshots/2e9cf7d297327f8c1890ac1616be62c87d6fe5f5
ImageFolder pronto em: data/fruits


In [6]:
def create_data_yaml(path_to_classes_txt, path_to_data_yaml):
  # Lê o arquivos "classes.txt".
  if not os.path.exists(path_to_classes_txt):
    print(f'classes.txt file not found! Please create a classes.txt labelmap and move it to {path_to_classes_txt}')
    return
  with open(path_to_classes_txt, 'r') as f:
    classes = []
    for line in f.readlines():
      if len(line.strip()) == 0: continue
      classes.append(line.strip())
  number_of_classes = len(classes)

  # Cria o dicionário a ser salvo.
  data = {
      'path': 'data/fruits',
      'train': 'images',
      'val': 'images',
      'nc': number_of_classes,
      'names': classes
  }

  # Escreve o arquivo YAML.
  with open(path_to_data_yaml, 'w') as f:
    yaml.dump(data, f, sort_keys=False)
  print(f'Created config file at {path_to_data_yaml}')

  return

# Chama a função.
create_data_yaml("data/fruits/classes.txt", "yolo_train.yaml")

Created config file at yolo_train.yaml


In [8]:
# Carrega o modelo pré-treinado.
model = YOLO("yolo11n.pt")

# Treina o modelo utilizando as informações do arquivo YAML.
# Definimos também a quantidade de épocas, o batch, e o tamanho das imagens.
results = model.train(data="./yolo_train.yaml", project="praticas/modulo_5/aula_1", epochs=10, batch=2, imgsz=480)

New https://pypi.org/project/ultralytics/8.3.243 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.240 🚀 Python-3.10.12 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 5804MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./yolo_train.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=480, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=Fals

In [9]:
# Video pode ser necessário alterar o path.
Video("video.mov")


In [10]:
model.predict("video.mov", save=True, project="praticas/modulo_5/aula_1")


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/239) /home/joaoferreira/trilha_visao_computacional/praticas/modulo_5/aula_1/video.mov: 288x480 4 apples, 1 carrot, 1 orange, 44.0ms
video 1/1 (frame 2/239) /home/joaoferreira/trilha_visao_computacional/praticas/modulo_5/aula_1/video.mov: 288x480 4 apples, 1 carrot, 1 orange, 4.1ms
video 1/1 (frame 3/239) /home/joaoferreira/trilha_visao_computacional/praticas/modulo_5/aula_1/video.mov: 288x480 3 apples, 1 carrot, 1 orange, 3.9ms
video 1

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'apple', 1: 'carrot', 2: 'orange'}
 obb: None
 orig_img: array([[[ 98,  86,  66],
         [ 98,  86,  66],
         [ 98,  86,  66],
         ...,
         [188, 177, 152],
         [192, 180, 158],
         [192, 180, 158]],
 
        [[ 98,  86,  66],
         [ 98,  86,  66],
         [ 98,  86,  66],
         ...,
         [188, 177, 152],
         [192, 180, 158],
         [192, 180, 158]],
 
        [[ 98,  86,  66],
         [ 98,  86,  66],
         [ 98,  86,  66],
         ...,
         [188, 177, 152],
         [192, 180, 158],
         [191, 179, 157]],
 
        ...,
 
        [[ 50,  51,  24],
         [ 50,  51,  24],
         [ 50,  52,  23],
         ...,
         [ 26, 133, 205],
         [ 27, 134, 206],
         [ 27, 134, 206]],
 
        [[ 50,  51,  24],
         [ 50,  51,  24],
         [ 50,  52,  23],
       

In [ ]:
Video("/home/joaoferreira/trilha_visao_computacional/praticas/modulo_5/aula_1/praticas/modulo_5/aula_1/predict/video.avi")

: 

## Próximos Passos e Referências

Nas próximas práticas vamos continuar trabalhando com problemas reais que envolvem Visão Computacional.

Uma lista não exaustiva de referências segue:

- https://docs.ultralytics.com/modes/train/
- https://docs.ultralytics.com/modes/predict/
- https://huggingface.co/datasets/johnatanvq/fruits-dataset
- https://huggingface.co/datasets
- https://pytorch.org/
- https://docs.pytorch.org/vision/main/models.html
- https://opencv.org/
- https://learnopencv.com/blogs/
- https://pyimagesearch.com/

## Atividades Complementares (Opcional)

- [ ] Tente alterar a base de dados e veja se o modelo continua funcionando?
- [ ] Tente alterar alguns hiperparâmetros de treinamento, batch e resolução da imagens e veja como isso altera os resultados.